In [2]:
# First, we access Python's settings with sys.
# The time module is used to measure how long the distribution function takes to run. 
import sys, time
sys.path.append("..")   # the notebook is inside notebooks/, so go up one folder to find src.
from src.combinatorics import omega, distribution # import our functions.
# We also import the statistics module to analyze the timing data.
import statistics

# 1) We test for the largest N whose peak Omega still fits in a float.
# Since plotting needs floats, we need to know how large N can be before the peak Omega exceeds the maximum float value.
# This loop keeps repeating until something inside stops it. Each pass tries the next N.
N = 1
while True:
    try:
        float(omega(N + 1, (N + 1) // 2))
        N += 1
    except OverflowError:
        break
print("Largest N whose Omega fits in a float:", N)

# 2) We measure how long the exact distribution takes as N grows.
# We repeat each timing several times, because the time varies a little between runs, and take the average and standard deviation of the times.
repeats = 5
for N in [100, 500, 1000, 2000, 4000]: # Outer loop over the different values of N.
    times = []
    for _ in range(repeats): # Inner loop to repeat the timing for the same N multiple times.
        t0 = time.perf_counter()
        distribution(N)
        times.append(time.perf_counter() - t0)
    mean = statistics.mean(times)
    std = statistics.stdev(times)
    print(f"N = {N:5d}: {mean:.4f} ± {std:.4f} s") # We print the result.

Largest N whose Omega fits in a float: 1029
N =   100: 0.0003 ± 0.0001 s
N =   500: 0.0125 ± 0.0005 s
N =  1000: 0.0734 ± 0.0048 s
N =  2000: 0.4496 ± 0.0043 s
N =  4000: 3.2413 ± 0.0350 s


In [1]:
#Part 1a: Enumeration for N=7
import os
os.chdir("..")
from src.microstates import generate_microstates
from src.microstates import group_by_heads

microstates = generate_microstates(7)

print("All microstates:")
print(microstates)

N=7
microstates_list = generate_microstates(N)
groups = group_by_heads(microstates)
print("\nTotal number of microstates:", len(microstates_list))
print("\nMicrostates grouped by number of heads:")

for k, states in groups.items():
    print(f"{k} heads: {states}")

All microstates:
['TTTTTTT', 'TTTTTTH', 'TTTTTHT', 'TTTTTHH', 'TTTTHTT', 'TTTTHTH', 'TTTTHHT', 'TTTTHHH', 'TTTHTTT', 'TTTHTTH', 'TTTHTHT', 'TTTHTHH', 'TTTHHTT', 'TTTHHTH', 'TTTHHHT', 'TTTHHHH', 'TTHTTTT', 'TTHTTTH', 'TTHTTHT', 'TTHTTHH', 'TTHTHTT', 'TTHTHTH', 'TTHTHHT', 'TTHTHHH', 'TTHHTTT', 'TTHHTTH', 'TTHHTHT', 'TTHHTHH', 'TTHHHTT', 'TTHHHTH', 'TTHHHHT', 'TTHHHHH', 'THTTTTT', 'THTTTTH', 'THTTTHT', 'THTTTHH', 'THTTHTT', 'THTTHTH', 'THTTHHT', 'THTTHHH', 'THTHTTT', 'THTHTTH', 'THTHTHT', 'THTHTHH', 'THTHHTT', 'THTHHTH', 'THTHHHT', 'THTHHHH', 'THHTTTT', 'THHTTTH', 'THHTTHT', 'THHTTHH', 'THHTHTT', 'THHTHTH', 'THHTHHT', 'THHTHHH', 'THHHTTT', 'THHHTTH', 'THHHTHT', 'THHHTHH', 'THHHHTT', 'THHHHTH', 'THHHHHT', 'THHHHHH', 'HTTTTTT', 'HTTTTTH', 'HTTTTHT', 'HTTTTHH', 'HTTTHTT', 'HTTTHTH', 'HTTTHHT', 'HTTTHHH', 'HTTHTTT', 'HTTHTTH', 'HTTHTHT', 'HTTHTHH', 'HTTHHTT', 'HTTHHTH', 'HTTHHHT', 'HTTHHHH', 'HTHTTTT', 'HTHTTTH', 'HTHTTHT', 'HTHTTHH', 'HTHTHTT', 'HTHTHTH', 'HTHTHHT', 'HTHTHHH', 'HTHHTTT', 'HT

In [ ]:
#Part 1(b) Maximum N for microstates enumeration
#The objective is to determine the max number of coins, N, for which our program can generate and store all possible microstates


import os
os.chdir("..")
import time
import gc

import psutil

from src.microstates import generate_microstates


def measure_microstates(N_values):

    process = psutil.Process(os.getpid())

    results = []

    for N in N_values:

        gc.collect()

        # Memory before generating microstates
        memory_before = process.memory_info().rss / (1024 ** 2)

        # Measure execution time
        start_time = time.perf_counter()

        microstates = generate_microstates(N)

        elapsed_time = time.perf_counter() - start_time

        # Memory after generating microstates
        memory_after = process.memory_info().rss / (1024 ** 2)

        memory_used = memory_after - memory_before

        results.append({
            "N": N,
            "Microstates": len(microstates),
            "Time (s)": elapsed_time,
            "Memory (MB)": memory_used
        })

        print(
            f"N={N} | "
            f"Microstates={len(microstates)} | "
            f"Time={elapsed_time:.4f}s | "
            f"Memory={memory_used:.2f} MB"
        )

        del microstates
        gc.collect()

    return results
N_values=[28]
results=measure_microstates(N_values)